In [7]:
# XML 아이템 하나를 처리하는 함수
def to_extract_row(item, tag_names, col_names):
    data = {}
    for tag, col in zip(tag_names, col_names):
        tag_obj = item.find(tag)
        data[col] = tag_obj.text.strip() if tag_obj else None
    return data

In [8]:
tag_names = ['bidCnt','pblancNo','pblancOdr','dcsNo','iemNo','bidNm','busiDivs','orntCode','ornt','opengDt','tbidCode','tbidName','tbidEname','tbidRptr','tbidTel','tbidAddr','tbidRate','tbidAmount']
col_names = ['참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '업무구분', '발주기관코드', '발주기관', '개찰일시', '낙찰자(업체코드)', '낙찰자(상호)', '낙찰자(영문상호)', '낙찰자(대표자)', '낙찰자(연락처)', '낙찰자(주소)', '낙찰률', '낙찰금액']
save_path = "workbookdata"

In [10]:
to_extract_row(soup.find_all('item')[0], tag_names, col_names)

{'참가수': '2',
 '공고번호': 'HDL0015',
 '공고차수': '1',
 '판단번호': '50495',
 '항목번호': '***',
 '입찰명': "'18년도 무한잉크프린터 임차용역",
 '업무구분': '용역',
 '발주기관코드': 'HDL',
 '발주기관': '공군제16전투비행단',
 '개찰일시': '201712291400',
 '낙찰자(업체코드)': 'H83BD',
 '낙찰자(상호)': '늑대와여우(현주컴퓨터)',
 '낙찰자(영문상호)': 'Wolf&Fox computer',
 '낙찰자(대표자)': '김성립',
 '낙찰자(연락처)': '054-654-0207',
 '낙찰자(주소)': '경상북도 예천군 예천읍 군청앞길9-0',
 '낙찰률': '96.7369',
 '낙찰금액': '24750000'}

In [27]:
import os
import time
import requests  
import pandas as pd
from bs4 import BeautifulSoup
from glob import glob

#API를 통해 XML data 불러오기
def request_estate_data(start_date, end_date, num_of_rows='15000'):
    serviceKey = '8bea1c13542aaa1626e136b275cde504cb37de3c3a31d19cd15d7759b330efdf'
    base_url = 'http://openapi.d2b.go.kr/openapi/service/BidResultInfoService'
    service_name = 'getDmstcSuccessBidResult'
    
    params = {
        'serviceKey': serviceKey,
        'opengDateBegin': start_date,
        'opengDateEnd': end_date,
        'numOfRows': num_of_rows,
        'pageNo': '1'
    }
    
    try:
        response = requests.get(base_url + service_name, params=params, timeout=30)
        response.raise_for_status()  # HTTP 오류가 발생하면 예외를 발생시킴
        soup = BeautifulSoup(response.text, 'xml')
        return soup
    except requests.exceptions.RequestException as e:
        print(f"{start_date[:4]}년 데이터 요청 중 오류 발생: {e}")
        return None

# XML 아이템 하나를 처리하는 함수
def to_extract_row(item, tag_names, col_names):
    data = {}
    for tag, col in zip(tag_names, col_names):
        tag_obj = item.find(tag)
        data[col] = tag_obj.text.strip() if tag_obj else None
    return data

# Soup 객체를 데이터프레임으로 변환하는 함수
def extract_data_and_DF(soup, tag_names, col_names):
    if not soup or not soup.find('item'):
        return pd.DataFrame() # 데이터가 없으면 빈 데이터프레임 반환
    
    item_list = [to_extract_row(item_tag, tag_names, col_names) for item_tag in soup.find_all('item')]
    return pd.DataFrame(item_list)

# 공통으로 사용할 변수들
tag_names = ['bidCnt','pblancNo','pblancOdr','dcsNo','iemNo','bidNm','busiDivs','orntCode','ornt','opengDt','tbidCode','tbidName','tbidEname','tbidRptr','tbidTel','tbidAddr','tbidRate','tbidAmount']
col_names = ['참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '업무구분', '발주기관코드', '발주기관', '개찰일시', '낙찰자(업체코드)', '낙찰자(상호)', '낙찰자(영문상호)', '낙찰자(대표자)', '낙찰자(연락처)', '낙찰자(주소)', '낙찰률', '낙찰금액']
save_path = "workbookdata"


if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2016년부터 2024년까지 반복
for year in range(2016, 2025):
    start_date = f'{year}0101'
    end_date = f'{year}1231'
    
    print(f"--- {year}년 데이터 처리 시작 ---")
    
    # 1. 데이터 요청
    soup = request_estate_data(start_date, end_date)
    
    # 2. 데이터프레임으로 변환
    if soup:
        df_year = extract_data_and_DF(soup, tag_names, col_names)
        
        # 3. CSV 파일로 즉시 저장
        if not df_year.empty:
            file_name = f'{year}년 물품 낙찰 결과.csv'
            full_path = os.path.join(save_path, file_name)
            df_year.to_csv(full_path, index=False, encoding='utf-8')
            print(f"{file_name} 저장 완료. (총 {len(df_year)}개 행)")
        else:
            print(f"{year}년 데이터는 비어있어 파일을 저장하지 않습니다.")
    
    # API 서버에 부담을 주지 않기 위해 요청 사이에 잠시 대기
    time.sleep(1) 

print("\n 모든 연도별 데이터 처리 완료!!")

--- 2016년 데이터 처리 시작 ---
2016년 물품 낙찰 결과.csv 저장 완료. (총 10513개 행)
--- 2017년 데이터 처리 시작 ---
2017년 물품 낙찰 결과.csv 저장 완료. (총 10249개 행)
--- 2018년 데이터 처리 시작 ---
2018년 물품 낙찰 결과.csv 저장 완료. (총 10390개 행)
--- 2019년 데이터 처리 시작 ---
2019년 물품 낙찰 결과.csv 저장 완료. (총 10631개 행)
--- 2020년 데이터 처리 시작 ---
2020년 물품 낙찰 결과.csv 저장 완료. (총 10193개 행)
--- 2021년 데이터 처리 시작 ---
2021년 물품 낙찰 결과.csv 저장 완료. (총 10122개 행)
--- 2022년 데이터 처리 시작 ---
2022년 물품 낙찰 결과.csv 저장 완료. (총 10457개 행)
--- 2023년 데이터 처리 시작 ---
2023년 물품 낙찰 결과.csv 저장 완료. (총 10518개 행)
--- 2024년 데이터 처리 시작 ---
2024년 물품 낙찰 결과.csv 저장 완료. (총 10145개 행)

 모든 연도별 데이터 처리 완료!!


In [13]:
df_year

,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,업무구분,발주기관코드,발주기관,개찰일시,낙찰자(업체코드),낙찰자(상호),낙찰자(영문상호),낙찰자(대표자),낙찰자(연락처),낙찰자(주소),낙찰률,낙찰금액
0,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,물품,SFL,제3283부대,201612301230,H84F8,주식회사 전우밀리터리,Jung woo military,이정구,0222730487,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,23689950
1,1,UMM0792,2,41871,***,"17년 직접입영장병 신체검사 용역(경기 연천,파주)",용역,UMM,국군재정관리단,201612301100,D2701,(사)대한결핵협회,knta,신민석,0226339461,서울특별시 서초구 바우뫼로6길57 (우면동),69.8055,45300000
2,3,MDN0001,1,00023,***,'17년 IP교환기 유지보수,용역,MDN,국군화생방방호사령부,201612301100,C980F,씨지에스아이티(주),CGSIT,최규식,0226328730,"서울특별시 금천구 가산디지털1로1-0 (가산동, 더루벤스밸리) 1208호",88.8825,21115480
3,2,UMM0829,1,45191,***,17년 직접입영장병 신체검사 용역(인천 부평등 3개소),용역,UMM,국군재정관리단,201612301100,D2701,(사)대한결핵협회,knta,신민석,0226339461,서울특별시 서초구 바우뫼로6길57 (우면동),74.1187,59572000
4,6,HDC0035,2,47685,***,17년 무한 임대프린터 계약,용역,HDC,공군제1전투비행단,201612301030,D9307,주식회사 광명프라자,None,하윤수,0622613353,광주광역시 북구 서하로 356-1(),90.0891,21980000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10508,3,MCT0031,1,45032,***,2016년 승강기 유지관리용역,용역,MCT,국군체육부대,201601041030,E81B2,청운엘리베이터주식회사,cheoungun E/V,황지성,0544623945,"경상북도 구미시 산동읍 신당1로4길5-4, 506호(이룸프라자2)",88.4676,18404000
10509,4,UAZ0065,1,45350,***,16-국방어학원 승강기 관리용역,용역,UAZ,합동군사대학교,201601041030,CD34A,신한엘리베이터(주),"Shinhan Elevator Co.,Ltd.",김수복,0319174796,경기도 고양시 덕양구 호수로71번길70-0 (토당동),88.0836,10181000
10510,5,LHQ0062,1,45448,***,16년 근무지원단 사업장 생활폐기물 처리 단가 계약,용역,LHQ,제2작전사령부,201601041030,F7AA5,주식회사 그린알앤이,green.co.kr,이근수,0532568272,대구광역시 중구 국채보상로142길33-27 (동인동4가),88.1043,21830964
10511,3,MDK0030,1,45645,***,15-4차 남수단 한빛부대 긴급재보급 항공수송 용역,용역,MDK,국군수송사령부,201601041030,KS05R,선진로지스틱스(주),"Sunjin Logistics Co.,Ltd",정유진,0222259600,"서울특별시 강동구 양재대로 1553, 선진빌딩4층(천호동)",88.3711,27149000


### 하나의 csv로 합치기

In [28]:
path = "workbookdata"
fname = '*물품 낙찰 결과.csv'
full_pattern = os.path.join(path, fname)
fileList = glob(full_pattern)
fileList

['workbookdata\\2016년 물품 낙찰 결과.csv',
 'workbookdata\\2017년 물품 낙찰 결과.csv',
 'workbookdata\\2018년 물품 낙찰 결과.csv',
 'workbookdata\\2019년 물품 낙찰 결과.csv',
 'workbookdata\\2020년 물품 낙찰 결과.csv',
 'workbookdata\\2021년 물품 낙찰 결과.csv',
 'workbookdata\\2022년 물품 낙찰 결과.csv',
 'workbookdata\\2023년 물품 낙찰 결과.csv',
 'workbookdata\\2024년 물품 낙찰 결과.csv']

In [30]:
import pandas as pd
from glob import glob
import os

all_data_frames = []

for file_path in fileList:
    df = pd.read_csv(file_path, header=0, encoding='utf-8')
    all_data_frames.append(df)

PBL_df = pd.concat(all_data_frames, ignore_index=True)
PBL_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93218 entries, 0 to 93217
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   참가수        93218 non-null  int64  
 1   공고번호       93218 non-null  object 
 2   공고차수       93218 non-null  int64  
 3   판단번호       93218 non-null  object 
 4   항목번호       93218 non-null  object 
 5   입찰명        93218 non-null  object 
 6   업무구분       93218 non-null  object 
 7   발주기관코드     93218 non-null  object 
 8   발주기관       93218 non-null  object 
 9   개찰일시       93218 non-null  int64  
 10  낙찰자(업체코드)  93218 non-null  object 
 11  낙찰자(상호)    93218 non-null  object 
 12  낙찰자(영문상호)  67962 non-null  object 
 13  낙찰자(대표자)   93218 non-null  object 
 14  낙찰자(연락처)   93218 non-null  object 
 15  낙찰자(주소)    93218 non-null  object 
 16  낙찰률        91973 non-null  float64
 17  낙찰금액       93218 non-null  float64
dtypes: float64(2), int64(3), object(13)
memory usage: 12.8+ MB


In [24]:
PBL_df = PBL_df[~PBL_df['업무구분'] != '용역']
PBL_df['개찰일시'] = pd.to_datetime(PBL_df['개찰일시'].astype(str), format='%Y%m%d%H%M')

In [31]:
PBL_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93218 entries, 0 to 93217
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   참가수        93218 non-null  int64  
 1   공고번호       93218 non-null  object 
 2   공고차수       93218 non-null  int64  
 3   판단번호       93218 non-null  object 
 4   항목번호       93218 non-null  object 
 5   입찰명        93218 non-null  object 
 6   업무구분       93218 non-null  object 
 7   발주기관코드     93218 non-null  object 
 8   발주기관       93218 non-null  object 
 9   개찰일시       93218 non-null  int64  
 10  낙찰자(업체코드)  93218 non-null  object 
 11  낙찰자(상호)    93218 non-null  object 
 12  낙찰자(영문상호)  67962 non-null  object 
 13  낙찰자(대표자)   93218 non-null  object 
 14  낙찰자(연락처)   93218 non-null  object 
 15  낙찰자(주소)    93218 non-null  object 
 16  낙찰률        91973 non-null  float64
 17  낙찰금액       93218 non-null  float64
dtypes: float64(2), int64(3), object(13)
memory usage: 12.8+ MB


In [32]:
PBL_df.to_csv(os.path.join(path, 'Total 물품 낙찰 결과.csv'), index=False, encoding='cp949')

In [33]:
df = pd.read_csv('workdata/Total 물품 낙찰 결과.csv', encoding='cp949')
df

FileNotFoundError: [Errno 2] No such file or directory: 'workdata/Total 물품 낙찰 결과.csv'

In [21]:
PBL_df = df.drop(columns='업무구분')
PBL_df

,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,발주기관코드,발주기관,개찰일시,낙찰자(업체코드),낙찰자(상호),낙찰자(영문상호),낙찰자(대표자),낙찰자(연락처),낙찰자(연락처).1,낙찰자(주소),낙찰률,낙찰금액
0,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,SFL,제3283부대,2016-12-30 12:30,H84F8,주식회사 전우밀리터리,Jung woo military,이정구,222730487,02-2273-0487,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,"23,689,950"
1,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,LFC,육군훈련소,2016-12-30 10:00,GA6E7,주식회사 엘엔에프 농업회사법인,LNF,이용환,437318109,043-731-8109,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,"5,810"
2,22,LGT0034,1,48136,***,0부대 중대시설 분전반 제조설치(16-312),LGT,제2307부대,2016-12-29 11:00,F91A4,설악테크,seorak tech,김화성,334610399,033-461-0399,강원특별자치도 인제군 북면 원통로74번길10-5(원통농공단지),88.0408,"29,098,720"
3,22,UMM0839,1,48049,***,13-본-국대-01 00학교 가구 및 비품 제조설치,UMM,국군재정관리단,2016-12-29 11:00,B30CC,주식회사 디에스나이키,"DS NAIKI CO.,LTD",김준홍,27342125,02-734-2125,"서울특별시 구로구 구로중앙로134, 4층(구로동, 리치몰)",88.2297,"2,165,042,000"
4,9,LGT0035,1,48138,***,00부대 중대시설 조명기구 제조납품(16-314),LGT,제2307부대,2016-12-29 11:00,C959B,주식회사 한서,"HANSEO Co.,LTD",이명옥,337462110,033-746-2110,강원특별자치도 원주시 태장공단길47-0 (태장동),88.3713,"32,424,410"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74727,164,SCQ0154,2,54928,***,수영장자동청소기 등 16종 구매,SCQ,제9911부대,2024-01-03 10:30,EF2CA,대광종합상사,DaKang a general trading company,김정현,336539838,033-653-9838,강원특별자치도 강릉시 경강로2496,89.4840,"25,489,200"
74728,33,HCL0005,1,51,***,24년 간부식당 식자재 구매,HCL,공군사관학교,2024-01-03 10:30,D3B5E,미가온,MIGAON,변정순,319438200,031-943-8200,경기도 파주시 장명산길174-20 (오도동),83.3270,"351,974,920"
74729,3,SCR0090,2,61362,***,24년 배추김치(포기절단) 등 10종 구매,SCR,해군제3함대사령부,2024-01-03 10:30,J47CB,현진식품 영농조합법인,NaN,정재웅,614539877,061-453-9877,전라남도 무안군 망운면 조금나루길22,88.9890,"53,896,000"
74730,31,MDS0030,1,66867,***,교육단 증개축 GHP 실외기 설치납품,MDS,제3707부대,2024-01-03 10:30,E1986,(주)비케이에너지,BKenergy,이복경,318377674,031-837-7674,경기도 의정부시 민락로387-0 (낙양동),88.0390,"60,496,000"


In [25]:
df[['입찰명']].head(15)

,입찰명
0,임시명찰 등 2품목 구매
1,장거리 이동 신병 도시락 제조납품
2,0부대 중대시설 분전반 제조설치(16-312)
3,13-본-국대-01 00학교 가구 및 비품 제조설치
4,00부대 중대시설 조명기구 제조납품(16-314)
5,2017년 난방연료
6,00부대 체육시설 비품구매(고소작업대 등 9종)
7,17년 액화석유가스(LPG) 구매
8,군수처 취사연료
9,우갈비 등 15품목 (덕산스포텔)
